# Regressions: Economic Complexity and Natural Resources

**Capstone Project — Resource Curse / ECI**  
*Pipeline step 6 — Regression Analysis*

Models:
- **Model 1** — Kitchen Sink (~44 vars), DV = log(ECI), SE clustered by K-means cluster
- **Model 2** — AR Baseline (ECI_{t-1} only), DV = ECI, SE clustered by country
- **Model 3a** — Base, no lag; DV = ECI
- **Model 3b** — Base + ECI_{t-1}; DV = ECI
- **Model 3c** — All regressors lagged; DV = ECI
- **Model 3d** — Extended controls + ECI_{t-1}; DV = ECI
- **Model 3e** — First differences (ΔECI)

SE estimator: clustered by country (except Model 1: clustered by K-means cluster).  
No country fixed effects.

## 0. Setup

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

OUT = 'Graphics/NB6'
os.makedirs(OUT, exist_ok=True)

ECI_COL = 'Economic Complexity Index'
FONT    = 'IBM Plex Sans, Arial, sans-serif'
BG      = '#fafafa'
NAVY    = '#1a1a2e'
GRID    = '#e0e0e0'

def save_html(fig, name):
    path = os.path.join(OUT, f'{name}.html')
    fig.write_html(path, config={'displayModeBar': False, 'responsive': True})
    print(f'Saved: {path}')

print('Setup complete')

Setup complete


## 1. Load Data

In [2]:
master   = pd.read_csv('intermediary/Master.csv', dtype={'Country Code': str})
clusters = pd.read_csv('intermediary/clusters_k5_agg.csv', dtype={'Country Code': str})

cluster_map = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
master = master.merge(cluster_map, on='Country Code', how='inner')

master['Year'] = master['Year'].astype(int)
master = master.sort_values(['Country Code', 'Year']).reset_index(drop=True)

print(f'Sample: {master["Country Code"].nunique()} countries, {len(master):,} obs')
print(f'Years: {master["Year"].min()}-{master["Year"].max()}')
print(f'\nCluster distribution:')
print(master.drop_duplicates('Country Code')['ClusterLabels'].value_counts())

Sample: 47 countries, 1,175 obs
Years: 1995-2019

Cluster distribution:
ClusterLabels
Oil Exporters              11
Low-Intensity Producers    11
Petrostates                11
Diversified Producers       8
Hard Mineral Exporters      6
Name: count, dtype: int64


## 2. Feature Engineering

In [3]:
df = master.copy()

df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan)
)
df['Total_Reserves_Value_Per_Capita'] = (
    df['Total_Reserves_Value'] / df['Population'].replace(0, np.nan)
)

df['log_HCI']              = np.log1p(df['Human capital index'].clip(lower=0))
df['log_GFCF']             = np.log1p(
    df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'].clip(lower=0))

eci_min = df[ECI_COL].min()
df['log_ECI'] = np.log(df[ECI_COL] - eci_min + 1)

df['ECI_lag1']  = df.groupby('Country Code')[ECI_COL].shift(1)
df['delta_ECI'] = df[ECI_COL] - df['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability — estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    df[f'{var}_lag1'] = df.groupby('Country Code')[var].shift(1)

hci_c  = df['log_HCI']             - df['log_HCI'].mean()
gfcf_c = df['log_GFCF']            - df['log_GFCF'].mean()
prod_c = df['log_Production_Value'] - df['log_Production_Value'].mean()
df['log_HCI_x_log_Production']  = hci_c  * prod_c
df['log_GFCF_x_log_Production'] = gfcf_c * prod_c

hci_l_c  = df['log_HCI_lag1']             - df['log_HCI_lag1'].mean()
gfcf_l_c = df['log_GFCF_lag1']            - df['log_GFCF_lag1'].mean()
prod_l_c = df['log_Production_Value_lag1'] - df['log_Production_Value_lag1'].mean()
df['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
df['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

print('Feature engineering complete')
print(f'Shape: {df.shape}')

# ── First differences for Model 3e ────────────────────────────────────────────
# A proper first-difference specification requires differencing both sides of
# the equation. Time-invariant variables (dominance dummies) drop out; only
# time-varying regressors and interactions are differenced.
for var in BASE_INDEP:
    df[f'd_{var}'] = df.groupby('Country Code')[var].diff()

df['d_Electricity'] = df.groupby('Country Code')['Access to electricity (% of population)'].diff()

# Difference the interaction terms directly (Δ(X·Z), not ΔX·ΔZ)
df['d_log_HCI_x_log_Production']  = df.groupby('Country Code')['log_HCI_x_log_Production'].diff()
df['d_log_GFCF_x_log_Production'] = df.groupby('Country Code')['log_GFCF_x_log_Production'].diff()


Feature engineering complete
Shape: (1175, 70)


## 3. Descriptive Statistics

In [4]:
VARIABLE_GROUPS = {
    'Economic Complexity': [ECI_COL],
    'Resource Dependence': [
        'Oil rents (% of GDP)', 'Natural gas rents (% of GDP)',
        'Mineral rents (% of GDP)', 'Total natural resources rents (% of GDP)',
        'Total_Production_Value_Per_Capita',
        'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
    ],
    'Economic Structure': [
        'Trade (% of GDP)', 'Agriculture', 'Industry', 'Manufacturing', 'Services',
        'Share of investment in GDP', 'Share of government spending in GDP',
        'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    ],
    'Macroeconomic': [
        'GDP per capita (constant prices, PPP)',
        'Inflation, consumer prices (annual %)', 'Real interest rate (%)',
        'Capital depreciation rate', 'Government revenue',
        'Use of IMF credit (DOD, current US$)',
    ],
    'Human Development': [
        'Human capital index', 'Life expectancy at birth, total (years)',
        'Urban population (% of total population)',
        'Access to electricity (% of population)',
        'Mobile cellular subscriptions (per 100 people)',
    ],
    'Governance': [
        'Political stability — estimate', 'Rule of law index',
        'Political corruption index', 'Property rights', 'Clientelism index',
    ],
    'Finance': [
        'Domestic credit to private sector (% of GDP)',
        'Adjusted savings: gross savings (% of GNI)',
        'Lending interest rate (%)',
        'Primary net lending, General government, Percent of GDP',
    ],
}

stats_rows = []
for group, vars_list in VARIABLE_GROUPS.items():
    for v in vars_list:
        if v not in df.columns:
            continue
        s = df[v].dropna()
        stats_rows.append({
            'Group': group, 'Variable': v,
            'Mean': round(s.mean(), 3), 'Std': round(s.std(), 3),
            'Median': round(s.median(), 3),
            'Q1': round(s.quantile(0.25), 3), 'Q3': round(s.quantile(0.75), 3),
            'N': len(s),
        })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv(os.path.join(OUT, 'descriptive_stats.csv'), index=False)

for group in VARIABLE_GROUPS:
    sub = stats_df[stats_df['Group'] == group]
    if len(sub) == 0:
        continue
    print(f'\n--- {group} ---')
    print(sub[['Variable','Mean','Std','Median','Q1','Q3','N']].to_string(index=False))


--- Economic Complexity ---
                 Variable   Mean   Std  Median     Q1    Q3    N
Economic Complexity Index -0.461 0.782  -0.446 -1.046 0.152 1175

--- Resource Dependence ---
                                Variable     Mean      Std  Median      Q1       Q3    N
                    Oil rents (% of GDP)   10.582   14.117   3.212   0.656   17.019 1175
            Natural gas rents (% of GDP)    0.948    1.481   0.274   0.005    1.156 1175
                Mineral rents (% of GDP)    1.202    2.785   0.130   0.007    0.963 1175
Total natural resources rents (% of GDP)   14.760   13.950   9.355   3.544   22.747 1175
       Total_Production_Value_Per_Capita 2297.691 6731.064 402.009 128.476 1520.844 1173
                   Hydrocarbons_Dominant    0.775    0.418   1.000   1.000    1.000 1167
                 Subsoil_Metals_Dominant    0.119    0.324   0.000   0.000    0.000 1167
                Precious_Metals_Dominant    0.063    0.244   0.000   0.000    0.000 1167

--- Econom

## 4. ECI Evolution

In [5]:
eci_stats = df.groupby('Year')[ECI_COL].agg(
    Mean='mean',
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).reset_index()

cluster_eci = df.groupby(['Year', 'ClusterLabels'])[ECI_COL].mean().reset_index()

CLUSTER_COLORS = {
    'Petrostates':             '#E63946',
    'Oil Exporters':           '#457B9D',
    'Diversified Producers':   '#2A9D8F',
    'Low-Intensity Producers': '#E9C46A',
    'Hard Mineral Exporters':  '#A8DADC',
}

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=eci_stats['Year'].tolist() + eci_stats['Year'].tolist()[::-1],
    y=eci_stats['Q3'].tolist() + eci_stats['Q1'].tolist()[::-1],
    fill='toself', fillcolor='rgba(26,26,46,0.10)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=True, name='IQR (full sample)', hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=eci_stats['Year'], y=eci_stats['Mean'],
    mode='lines+markers',
    line=dict(color=NAVY, width=2.5),
    marker=dict(size=5, color=NAVY),
    name='Sample mean',
))
for lbl, color in CLUSTER_COLORS.items():
    sub = cluster_eci[cluster_eci['ClusterLabels'] == lbl]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=sub['Year'], y=sub[ECI_COL],
        mode='lines', line=dict(color=color, width=1.5, dash='dot'),
        name=lbl,
    ))

fig.update_layout(
    font=dict(family=FONT, color=NAVY),
    paper_bgcolor=BG, plot_bgcolor=BG,
    width=1000, height=450,
    margin=dict(l=60, r=200, t=40, b=50),
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='ECI', gridcolor=GRID, gridwidth=0.5),
    legend=dict(
        x=1.01, y=0.5, xanchor='left', yanchor='middle',
        font=dict(size=11), bgcolor='rgba(250,250,250,0.9)',
        bordercolor='#ccc', borderwidth=1,
    ),
)
save_html(fig, 'eci_evolution')
fig.show()

Saved: Graphics/NB6/eci_evolution.html


## 5. ECI Correlation Heatmap

In [6]:
CORR_VARS = [
    'Total_Production_Value_Per_Capita', 'Oil rents (% of GDP)',
    'Natural gas rents (% of GDP)', 'Mineral rents (% of GDP)',
    'Total natural resources rents (% of GDP)',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
    'Trade (% of GDP)', 'GDP per capita (constant prices, PPP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP', 'Agriculture', 'Industry', 'Manufacturing', 'Services',
    'Inflation, consumer prices (annual %)', 'Real interest rate (%)',
    'Domestic credit to private sector (% of GDP)',
    'Adjusted savings: gross savings (% of GNI)',
    'Human capital index', 'Life expectancy at birth, total (years)',
    'Urban population (% of total population)',
    'Access to electricity (% of population)',
    'Mobile cellular subscriptions (per 100 people)',
    'Political stability — estimate', 'Rule of law index',
    'Political corruption index', 'Property rights',
    'Government revenue', 'Use of IMF credit (DOD, current US$)',
    'Landlocked', 'Capital depreciation rate',
]
CORR_VARS = [v for v in CORR_VARS if v in df.columns]

corr_vals = df[[ECI_COL] + CORR_VARS].corr()[ECI_COL].drop(ECI_COL)
corr_df   = corr_vals.reset_index().rename(columns={'index': 'Variable', ECI_COL: 'Corr'})
corr_df   = corr_df.sort_values('Corr', ascending=True)

z = corr_df['Corr'].values.reshape(-1, 1)
y = corr_df['Variable'].tolist()

fig = go.Figure(go.Heatmap(
    z=z, y=y, x=['Correlation with ECI'],
    colorscale=[
        [0.0, '#D73027'], [0.25, '#FC8D59'], [0.5, '#FFFFBF'],
        [0.75, '#91BFDB'], [1.0, '#4575B4'],
    ],
    zmid=0, zmin=-1, zmax=1,
    text=np.round(z, 3), texttemplate='%{text}',
    textfont=dict(size=9),
    colorbar=dict(title='Corr', thickness=14, len=0.7),
    hovertemplate='<b>%{y}</b><br>Corr = %{z:.3f}<extra></extra>',
))
fig.update_layout(
    font=dict(family=FONT, color=NAVY),
    paper_bgcolor=BG,
    height=max(700, len(y) * 18 + 100),
    width=700,
    margin=dict(l=400, r=120, t=40, b=50),
)
save_html(fig, 'eci_correlation')
fig.show()

print('\nTop 10 positive:')
print(corr_df.nlargest(10, 'Corr')[['Variable', 'Corr']].to_string(index=False))
print('\nTop 10 negative:')
print(corr_df.nsmallest(10, 'Corr')[['Variable', 'Corr']].to_string(index=False))

Saved: Graphics/NB6/eci_correlation.html



Top 10 positive:
                                    Variable     Corr
     Access to electricity (% of population) 0.649681
                         Human capital index 0.637295
     Life expectancy at birth, total (years) 0.555480
Domestic credit to private sector (% of GDP) 0.553686
                                    Services 0.469786
                               Manufacturing 0.417635
                             Property rights 0.408435
                           Rule of law index 0.391264
       GDP per capita (constant prices, PPP) 0.372905
              Political stability — estimate 0.333340

Top 10 negative:
                                  Variable      Corr
                               Agriculture -0.456632
                Political corruption index -0.407676
  Total natural resources rents (% of GDP) -0.325506
                  Mineral rents (% of GDP) -0.211086
                      Oil rents (% of GDP) -0.192150
                  Precious_Metals_Dominant -0.148830

## 6. Regression Helpers

In [7]:
def run_ols(df_in, dv, regressors, cluster_by):
    req = [dv] + regressors + [cluster_by]
    sub = df_in.dropna(subset=req).copy()
    X   = sm.add_constant(sub[regressors])
    y   = sub[dv]
    res = sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': sub[cluster_by]},
    )
    return res, sub


def stars(p):
    if p < 0.01:  return '***'
    if p < 0.05:  return '**'
    if p < 0.10:  return '*'
    return ''


def format_table(res, title=''):
    tbl = pd.DataFrame({
        'Variable': res.params.index,
        'Coef':     res.params.values,
        'SE':       res.bse.values,
        'p':        res.pvalues.values,
        'CI_lo':    res.conf_int().iloc[:, 0].values,
        'CI_hi':    res.conf_int().iloc[:, 1].values,
    })
    tbl['Stars'] = tbl['p'].apply(stars)
    print(f'\n{title}  N={int(res.nobs):,}  R2={res.rsquared:.4f}')
    print(tbl[['Variable', 'Coef', 'SE', 'p', 'Stars']].to_string(
        index=False, float_format='{:.4f}'.format))
    return tbl


def save_coef_csv(tbl, name):
    path = os.path.join(OUT, f'{name}.csv')
    tbl.to_csv(path, index=False)
    print(f'CSV: {path}')


ALL_RESULTS = {}
print('Helpers defined')

Helpers defined


## 7. Model 1 - Kitchen Sink

DV = log(ECI). SE clustered by K-means cluster. Exploratory, intentionally over-fitted.

In [8]:
KITCHEN_SINK_VARS = [
    'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)',
    'Agriculture',
    'Capital depreciation rate',
    'Clientelism index',
    'Death rates, crude per 1000 people',
    'Domestic credit to private sector (% of GDP)',
    'GDP per capita (constant prices, PPP)',
    'Government revenue',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Human capital index',
    'Industry',
    'Inflation, consumer prices (annual %)',
    'Landlocked',
    'Lending interest rate (%)',
    'Life expectancy at birth, total (years)',
    'Manufacturing',
    'Mineral rents (% of GDP)',
    'Mobile cellular subscriptions (per 100 people)',
    'Natural gas rents (% of GDP)',
    'Oil rents (% of GDP)',
    'Political corruption index',
    'Political stability — estimate',
    'Primary net lending, General government, Percent of GDP',
    'Property rights',
    'Real interest rate (%)',
    'Rule of law index',
    'Services',
    'Share of consumption in GDP',
    'Share of government spending in GDP',
    'Share of investment in GDP',
    'Total natural resources rents (% of GDP)',
    'Trade (% of GDP)',
    'Urban population (% of total population)',
    'Use of IMF credit (DOD, current US$)',
    'Total_Production_Value_Per_Capita',
    'Total_Reserves_Value_Per_Capita',
    'Hydrocarbons_Dominant',
    'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
]
KITCHEN_SINK_VARS = [v for v in KITCHEN_SINK_VARS if v in df.columns]
print(f'Kitchen sink: {len(KITCHEN_SINK_VARS)} regressors')

res_m1, _ = run_ols(df, 'log_ECI', KITCHEN_SINK_VARS, 'Cluster')
tbl_m1 = format_table(res_m1, 'Model 1 - Kitchen Sink')
save_coef_csv(tbl_m1, 'coef_model1')
ALL_RESULTS['Model 1'] = (res_m1, tbl_m1)

Kitchen sink: 40 regressors

Model 1 - Kitchen Sink  N=1,117  R2=0.7328
                                                           Variable    Coef     SE      p Stars
                                                              const  0.8187 0.3766 0.0297    **
                            Access to electricity (% of population)  0.0031 0.0004 0.0000   ***
                         Adjusted savings: gross savings (% of GNI) -0.0022 0.0017 0.1821      
                                                        Agriculture -0.0057 0.0027 0.0329    **
                                          Capital depreciation rate -0.9167 1.2435 0.4610      
                                                  Clientelism index -0.0384 0.0858 0.6543      
                                 Death rates, crude per 1000 people  0.0029 0.0050 0.5645      
                       Domestic credit to private sector (% of GDP)  0.0016 0.0005 0.0035   ***
                              GDP per capita (constant prices, P

## 8. Model 2 - AR Baseline

DV = ECI. IV = ECI_{t-1} only. SE clustered by country. Benchmark (expected R2 ~ 0.97).

In [9]:
res_m2, _ = run_ols(df, ECI_COL, ['ECI_lag1'], 'Country Code')
tbl_m2 = format_table(res_m2, 'Model 2 - AR Baseline')
save_coef_csv(tbl_m2, 'coef_model2')
ALL_RESULTS['Model 2'] = (res_m2, tbl_m2)


Model 2 - AR Baseline  N=1,128  R2=0.8748
Variable    Coef     SE      p Stars
   const -0.0266 0.0153 0.0829     *
ECI_lag1  0.9389 0.0214 0.0000   ***
CSV: Graphics/NB6/coef_model2.csv


## 9. Model 3a - Base (no lag)

Main interpretive model. DV = ECI. SE clustered by country.

In [10]:
VARS_3A = BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production']

res_3a, _ = run_ols(df, ECI_COL, VARS_3A, 'Country Code')
tbl_3a = format_table(res_3a, 'Model 3a - Base (no lag)')
save_coef_csv(tbl_3a, 'coef_model3a')
ALL_RESULTS['3a'] = (res_3a, tbl_3a)


Model 3a - Base (no lag)  N=1,173  R2=0.4248
                      Variable    Coef     SE      p Stars
                         const -4.2690 0.5897 0.0000   ***
                       log_HCI  2.9011 0.4431 0.0000   ***
                      log_GFCF  0.0684 0.0821 0.4048      
Political stability — estimate -0.0179 0.0883 0.8391      
             Rule of law index  0.2987 0.3031 0.3244      
          log_Production_Value -0.0023 0.0374 0.9514      
              Trade (% of GDP)  0.0005 0.0024 0.8395      
      log_HCI_x_log_Production -0.1312 0.1721 0.4458      
     log_GFCF_x_log_Production  0.0015 0.0406 0.9705      
CSV: Graphics/NB6/coef_model3a.csv


## 10. Model 3b - One Lag

DV = ECI. Model 3a + ECI_{t-1}.

In [11]:
VARS_3B = BASE_INDEP + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']

res_3b, _ = run_ols(df, ECI_COL, VARS_3B, 'Country Code')
tbl_3b = format_table(res_3b, 'Model 3b - One Lag')
save_coef_csv(tbl_3b, 'coef_model3b')
ALL_RESULTS['3b'] = (res_3b, tbl_3b)


Model 3b - One Lag  N=1,126  R2=0.8773
                      Variable    Coef     SE      p Stars
                         const -0.4072 0.1578 0.0099   ***
                       log_HCI  0.2663 0.1238 0.0314    **
                      log_GFCF  0.0204 0.0135 0.1307      
Political stability — estimate  0.0113 0.0152 0.4546      
             Rule of law index  0.0200 0.0404 0.6204      
          log_Production_Value -0.0033 0.0049 0.5074      
              Trade (% of GDP)  0.0000 0.0004 0.9521      
                      ECI_lag1  0.8957 0.0369 0.0000   ***
      log_HCI_x_log_Production -0.0302 0.0220 0.1707      
     log_GFCF_x_log_Production  0.0058 0.0081 0.4751      
CSV: Graphics/NB6/coef_model3b.csv


## 11. Model 3c - All Regressors Lagged

DV = ECI. All base regressors + interactions at t-1 + ECI_{t-1}.

In [12]:
LAGGED_BASE = [f'{v}_lag1' for v in BASE_INDEP]
VARS_3C = LAGGED_BASE + [
    'ECI_lag1',
    'log_HCI_x_log_Production_lag1',
    'log_GFCF_x_log_Production_lag1',
]

res_3c, _ = run_ols(df, ECI_COL, VARS_3C, 'Country Code')
tbl_3c = format_table(res_3c, 'Model 3c - All Lagged')
save_coef_csv(tbl_3c, 'coef_model3c')
ALL_RESULTS['3c'] = (res_3c, tbl_3c)


Model 3c - All Lagged  N=1,126  R2=0.8771
                           Variable    Coef     SE      p Stars
                              const -0.3801 0.1712 0.0264    **
                       log_HCI_lag1  0.2300 0.1187 0.0527     *
                      log_GFCF_lag1  0.0090 0.0107 0.4019      
Political stability — estimate_lag1  0.0046 0.0103 0.6582      
             Rule of law index_lag1  0.0355 0.0384 0.3560      
          log_Production_Value_lag1  0.0005 0.0043 0.9118      
              Trade (% of GDP)_lag1  0.0003 0.0003 0.3418      
                           ECI_lag1  0.8982 0.0368 0.0000   ***
      log_HCI_x_log_Production_lag1 -0.0189 0.0193 0.3265      
     log_GFCF_x_log_Production_lag1 -0.0011 0.0055 0.8457      
CSV: Graphics/NB6/coef_model3c.csv


## 12. Model 3d - Extended Controls

DV = ECI. Model 3b + electricity + 3 resource-type dummies.

In [13]:
VARS_3D = BASE_INDEP + EXTRA_CONTROLS + [
    'ECI_lag1',
    'log_HCI_x_log_Production',
    'log_GFCF_x_log_Production',
]

res_3d, _ = run_ols(df, ECI_COL, VARS_3D, 'Country Code')
tbl_3d = format_table(res_3d, 'Model 3d - Extended')
save_coef_csv(tbl_3d, 'coef_model3d')
ALL_RESULTS['3d'] = (res_3d, tbl_3d)


Model 3d - Extended  N=1,120  R2=0.8802
                               Variable    Coef     SE      p Stars
                                  const -0.4493 0.1566 0.0041   ***
                                log_HCI  0.1643 0.1056 0.1197      
                               log_GFCF  0.0184 0.0161 0.2528      
         Political stability — estimate  0.0133 0.0197 0.4989      
                      Rule of law index  0.0485 0.0423 0.2519      
                   log_Production_Value -0.0153 0.0096 0.1087      
                       Trade (% of GDP)  0.0002 0.0004 0.5726      
                  Hydrocarbons_Dominant  0.0252 0.0396 0.5253      
                Subsoil_Metals_Dominant -0.0184 0.0458 0.6878      
               Precious_Metals_Dominant -0.0002 0.0423 0.9953      
Access to electricity (% of population)  0.0022 0.0009 0.0159    **
                               ECI_lag1  0.8601 0.0462 0.0000   ***
               log_HCI_x_log_Production -0.0112 0.0257 0.6623      
       

## 13. Model 3e - First Differences (delta ECI)

DV = delta_ECI. All time-varying regressors are first-differenced.
Time-invariant variables (dominance dummies) drop out. ECI_{t-1} enters
as an error-correction term. This tests whether *changes* in human capital,
investment, and institutions predict *changes* in economic complexity.


In [14]:
# ── Model 3e — First Differences (both sides differenced) ────────────────────
# Previous version only differenced the DV (delta_ECI = ECI - ECI_lag1) while
# keeping regressors in levels. That is algebraically equivalent to Model 3d
# with the ECI_lag1 coefficient shifted by -1 — it adds no new information.
#
# The corrected version differences all time-varying regressors. Time-invariant
# controls (dominance dummies) are absorbed by differencing and excluded.
# ECI_lag1 is retained as an error-correction term.

VARS_3E_FD = [f'd_{v}' for v in BASE_INDEP] + [
    'd_Electricity',
    'ECI_lag1',
    'd_log_HCI_x_log_Production',
    'd_log_GFCF_x_log_Production',
]

res_3e, _ = run_ols(df, 'delta_ECI', VARS_3E_FD, 'Country Code')
tbl_3e = format_table(res_3e, 'Model 3e - First Differences')
save_coef_csv(tbl_3e, 'coef_model3e')
ALL_RESULTS['3e'] = (res_3e, tbl_3e)


## 14. Summary Table (Models 3a-3e)

In [15]:
DISPLAY_LABELS = {
    'const':                                       'Constant',
    'log_HCI':                                     'Human Capital (log)',
    'log_GFCF':                                    'GFCF (log)',
    'Political stability — estimate':              'Political Stability',
    'Rule of law index':                           'Rule of Law',
    'log_Production_Value':                        'NR Production (log, pc)',
    'Trade (% of GDP)':                            'Trade (% GDP)',
    'log_HCI_x_log_Production':                    'HCI x Production',
    'log_GFCF_x_log_Production':                   'GFCF x Production',
    'ECI_lag1':                                    'ECI (t-1)',
    'log_HCI_lag1':                                'Human Capital (t-1)',
    'log_GFCF_lag1':                               'GFCF (t-1)',
    'Political stability — estimate_lag1':         'Political Stability (t-1)',
    'Rule of law index_lag1':                      'Rule of Law (t-1)',
    'log_Production_Value_lag1':                   'NR Production (t-1)',
    'Trade (% of GDP)_lag1':                       'Trade (t-1)',
    'log_HCI_x_log_Production_lag1':               'HCI x Production (t-1)',
    'log_GFCF_x_log_Production_lag1':              'GFCF x Production (t-1)',
    'Hydrocarbons_Dominant':                       'Hydrocarbons dominant',
    'Subsoil_Metals_Dominant':                     'Subsoil metals dominant',
    'Precious_Metals_Dominant':                    'Precious metals dominant',
    'd_log_HCI':                                    'Δ Human Capital (log)',
    'd_log_GFCF':                                    'Δ GFCF (log)',
    'd_Political stability — estimate':              'Δ Political Stability',
    'd_Rule of law index':                           'Δ Rule of Law',
    'd_log_Production_Value':                        'Δ NR Production (log, pc)',
    'd_Trade (% of GDP)':                            'Δ Trade (% GDP)',
    'd_Electricity':                                 'Δ Electricity Access',
    'd_log_HCI_x_log_Production':                    'Δ HCI × Production',
    'd_log_GFCF_x_log_Production':                   'Δ GFCF × Production',
    'Access to electricity (% of population)':     'Electricity access',
}

MODEL_ORDER  = ['3a', '3b', '3c', '3d', '3e']
MODEL_LABELS = {
    '3a': '3a Base', '3b': '3b + Lag',
    '3c': '3c All Lagged', '3d': '3d Extended', '3e': '3e dECI',
}

all_vars = [
    'log_HCI', 'log_GFCF', 'Political stability — estimate',
    'Rule of law index', 'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'log_HCI_lag1', 'log_GFCF_lag1',
    'Political stability — estimate_lag1', 'Rule of law index_lag1',
    'log_Production_Value_lag1', 'Trade (% of GDP)_lag1',
    'log_HCI_x_log_Production_lag1', 'log_GFCF_x_log_Production_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant', 'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
    'const',
]

rows = []
for var in all_vars:
    row = {'Variable': DISPLAY_LABELS.get(var, var)}
    for m in MODEL_ORDER:
        col_lbl = MODEL_LABELS[m]
        if m not in ALL_RESULTS:
            row[col_lbl] = ''
            continue
        _, tbl = ALL_RESULTS[m]
        match = tbl[tbl['Variable'] == var]
        if len(match) > 0:
            c, s = match.iloc[0]['Coef'], match.iloc[0]['Stars']
            row[col_lbl] = f'{c:.4f}{s}'
        else:
            row[col_lbl] = '-'
    rows.append(row)

for stat_lbl in ['N', 'R2']:
    row = {'Variable': stat_lbl}
    for m in MODEL_ORDER:
        col_lbl = MODEL_LABELS[m]
        if m not in ALL_RESULTS:
            row[col_lbl] = ''
            continue
        res, _ = ALL_RESULTS[m]
        row[col_lbl] = f'{int(res.nobs):,}' if stat_lbl == 'N' else f'{res.rsquared:.4f}'
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(OUT, 'regression_summary.csv'), index=False)
print(summary_df.to_string(index=False))

## 15. Coefficient Forest Plot (Models 3a-3e)

In [16]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF', 'Political stability — estimate',
    'Rule of law index', 'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant',
    'Access to electricity (% of population)',
]

# Model 3e uses first-differenced regressors — its coefficients measure the
# effect of *changes*, not levels, so it is excluded from this comparison.
FOREST_ORDER = [m for m in MODEL_ORDER if m != '3e']
SPEC_COLORS = ['#4a6fa5', '#c23a3a', '#2e7d4a', '#d4853b', '#7b6fa5']

y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {v: i for i, v in enumerate(y_labels)}
n_specs  = len(FOREST_ORDER)
offsets  = np.linspace(-0.25, 0.25, n_specs)

fig = go.Figure()
fig.add_vline(x=0, line=dict(color='#888', width=1.2, dash='dash'))

for i, m in enumerate(FOREST_ORDER):
    if m not in ALL_RESULTS:
        continue
    _, tbl = ALL_RESULTS[m]
    col = SPEC_COLORS[i]
    lbl = MODEL_LABELS[m]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched = matched.assign(
        Label=matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    )
    matched = matched.assign(y=matched['Label'].map(y_pos) + offsets[i])

    fig.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y'],
        mode='markers',
        marker=dict(size=8, color=col, line=dict(color='white', width=1.2)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=1.8, width=4,
        ),
        name=lbl,
        hovertemplate='%{text}<br>b=%{x:.4f}<extra>' + lbl + '</extra>',
        text=matched['Label'],
    ))

n_vars = len(y_labels)
fig.update_layout(
    font=dict(family=FONT, color=NAVY),
    paper_bgcolor=BG, plot_bgcolor=BG,
    width=1100, height=max(500, n_vars * 55 + 200),
    margin=dict(l=60, r=60, t=60, b=60),
    xaxis=dict(
        title='Coefficient (95% CI, clustered SE by country)',
        gridcolor=GRID, gridwidth=0.5, zeroline=False,
    ),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11),
        showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.6, n_vars - 0.4],
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='center', x=0.5, font=dict(size=10),
        bgcolor='rgba(255,255,255,0.0)',
    ),
)
save_html(fig, 'coef_forest_3a_3e')
fig.show()

## Done

All outputs in `Graphics/NB6/`.